### 🔧 Setup & Configuration
This cell imports all the necessary Python libraries for data handling, preprocessing, model training, and evaluation.  
We also define configuration variables such as:
- **Random seed** for reproducibility.  
- **Train/test split ratio** to control dataset partitioning.  
- **File paths** for input (raw dataset) and output (cleaned dataset).  

This ensures that the pipeline runs consistently and results can be replicated.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import os

# Configuration
RANDOM_SEED = 42
TRAIN_RATIO = 0.8
INPUT_FILE = "../data/raw/HealthConnect_Appointment_Data.csv"
OUTPUT_FILE = "../data/processed/cleaned.csv"

np.random.seed(RANDOM_SEED)


### 📂 Load & Validate Dataset
We load the raw appointment dataset using Pandas.  
We then validate the schema against the **HealthConnect Data Dictionary** to ensure all expected columns are present.  

If any required columns are missing, the pipeline raises an error.  
This step provides **basic error handling** and ensures data integrity before preprocessing.


In [2]:
df = pd.read_csv(INPUT_FILE)

expected_columns = [
    "appointment_id","patient_id","gender","age","age_group","appointment_type",
    "booking_date","appointment_date","appointment_day","appointment_time",
    "booking_lead_days","previous_appointments","previous_no_shows",
    "reminder_sent","reminder_channel","distance_to_clinic_km",
    "waiting_time_minutes","appointment_outcome"
]

missing_cols = set(expected_columns) - set(df.columns)
if missing_cols:
    raise ValueError(f"Schema mismatch! Missing columns: {missing_cols}")
print("✅ Schema validated successfully.")


✅ Schema validated successfully.


### 🧹 Preprocessing Functions
We define reusable functions for data preprocessing:
- **`clean_missing_values`**: Handles missing values in `distance_to_clinic_km` and `waiting_time_minutes` using median imputation.  
- **`encode_categorical`**: Converts categorical variables (e.g., gender, appointment type, reminder channel) into numerical format using one-hot encoding.  
- **`scale_features`**: Normalizes numerical features (age, distance, waiting time, booking lead days) using StandardScaler.  

These functions mirror what would be stored in `src/preprocessing.py` for modularity and reusability.


In [3]:
def clean_missing_values(df):
    df["distance_to_clinic_km"].fillna(df["distance_to_clinic_km"].median(), inplace=True)
    df["waiting_time_minutes"].fillna(df["waiting_time_minutes"].median(), inplace=True)
    return df

def encode_categorical(df):
    categorical_cols = ["gender","appointment_type","reminder_channel","appointment_time","appointment_day"]
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    return df

def scale_features(df):
    numeric_cols = ["age","distance_to_clinic_km","waiting_time_minutes","booking_lead_days"]
    scaler = StandardScaler()
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    return df


### ⚙️ Apply Preprocessing
We apply the preprocessing pipeline to the dataset:
1. Handle missing values.  
2. Encode categorical variables.  
3. Scale numerical features.  

The cleaned dataset is then saved to `/data/processed/cleaned.csv`.  
This file is **physical evidence** that preprocessing has been executed successfully.


In [4]:
df = clean_missing_values(df)
df = encode_categorical(df)
df = scale_features(df)

os.makedirs("../data/processed", exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False)
print("✅ Cleaned dataset saved to processed folder.")


✅ Cleaned dataset saved to processed folder.


### 🤖 Model Workflow
We define the target variable (`appointment_outcome`) and features.  
The dataset is split into training and testing sets (80/20).  

Three models are trained:
- **Logistic Regression** (baseline, interpretable).  
- **Random Forest** (ensemble, robust).  
- **XGBoost** (gradient boosting, high performance).  

This step provides **evidence of model integration** into the pipeline.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Target encoding
y = df["appointment_outcome"].apply(lambda x: 1 if str(x).strip().lower() == "no-show" else 0)

# Drop unused columns
drop_cols = [col for col in ["appointment_outcome", "appointment_id", "patient_id"] if col in df.columns]
X = df.drop(columns=drop_cols)

# One-hot encode categorical features
X_encoded = pd.get_dummies(X, drop_first=True)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=1-TRAIN_RATIO, random_state=RANDOM_SEED
)

# Baseline Logistic Regression
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)

# Random Forest
rf = RandomForestClassifier(random_state=RANDOM_SEED)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# XGBoost
xgb = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=RANDOM_SEED)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)


### 📊 Evaluation Metrics
We evaluate each model using:
- Accuracy  
- Precision  
- Recall  
- F1-score  
- ROC-AUC  
- Confusion Matrix  

These metrics provide a comprehensive view of model performance.  
The printed outputs serve as **direct evidence** of how well each model predicts appointment no-shows.


In [6]:
def evaluate_model(name, y_true, y_pred):
    print(f"\n{name} Results:")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1-score:", f1_score(y_true, y_pred))
    print("ROC-AUC:", roc_auc_score(y_true, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

evaluate_model("Logistic Regression", y_test, y_pred_lr)
evaluate_model("Random Forest", y_test, y_pred_rf)
evaluate_model("XGBoost", y_test, y_pred_xgb)



Logistic Regression Results:
Accuracy: 0.561
Precision: 0.5435684647302904
Recall: 0.5446985446985447
F1-score: 0.5441329179646937
ROC-AUC: 0.5604032222529334
Confusion Matrix:
 [[299 220]
 [219 262]]

Random Forest Results:
Accuracy: 0.615
Precision: 0.6116279069767442
Recall: 0.5467775467775468
F1-score: 0.5773874862788145
ROC-AUC: 0.612502453542916
Confusion Matrix:
 [[352 167]
 [218 263]]

XGBoost Results:
Accuracy: 0.595
Precision: 0.5833333333333334
Recall: 0.553014553014553
F1-score: 0.567769477054429
ROC-AUC: 0.5934629605149836
Confusion Matrix:
 [[329 190]
 [215 266]]


### 💾 Save Models
Trained models are saved into the `/models/` directory:
- Logistic Regression → `log_reg.pkl`  
- Random Forest → `random_forest.pkl`  
- XGBoost → `xgboost.pkl`  

These saved files are **artefacts proving implementation**.  
They can be reused later for deployment, retraining, or further evaluation.


In [7]:
import joblib
os.makedirs("../models", exist_ok=True)

joblib.dump(log_reg, "../models/log_reg.pkl")
joblib.dump(rf, "../models/random_forest.pkl")
joblib.dump(xgb, "../models/xgboost.pkl")

print("✅ Models saved successfully.")


✅ Models saved successfully.
